# Verification of simulation subpackage: Quality class

Quality class allows to simulate spectra and calculate integral quantities for a x-ray radiation quality

In [ ]:
import numpy as np
import pandas as pd
from scipy.interpolate import Akima1DInterpolator
import matplotlib.pyplot as plt
import warnings
import spekpy as sp
import uspekpy as usp
from metpyx.data import Qualities, OperationalQuantities
from metpyx.sim import Quality

## Reference case

Reference case for verification: N60 radiation quality, H*(10) operational quantity at 0º irradiation angle, mass energy transfer coefficients for air from PENELOPE 2018, air kerma to dose conversion coefficients from CMI 2025, measurement distance at 1 m, air thickness equal to distance.

### Independent calculation

In [ ]:
# Independent calculation

# Define coefficients data
pene_2018_energies = [
    1.0, 1.1726, 1.25, 1.4, 1.5, 1.75, 2.0, 2.5, 3.0, 3.2063, 3.206301, 3.22391, 3.25051, 3.5, 3.61881, 4.0,
    5.0, 6.0, 7.0, 8.0, 9.0, 10.0, 12.5, 14.0, 15.0, 17.5, 20.0, 25.0, 28.6633, 30.0, 35.0, 40.0, 50.0, 60.0,
    70.0, 80.0, 90.0, 100.0, 125.0, 140.0, 150.0, 175.0, 187.083, 200.0, 250.0, 300.0, 324.037, 350.0, 386.867,
    400.0, 474.342, 500.0, 574.456, 600.0, 673.537, 700.0, 800.0, 900.0, 1000.0, 1250.0, 1500.0, 1558.93,
    1750.0, 1870.83, 2000.0, 2345.21, 2500.0, 3000.0, 3240.37, 3500.0, 4000.0, 4500.0, 5000.0, 6000.0, 6480.74,
    7000.0, 8000.0, 9000.0
]
pene_2018_values = [
    3487.7, 2271.66, 1907.85, 1396.25, 1152.41, 746.85, 510.495, 267.712, 156.677, 128.597, 139.322, 139.22,
    136.749, 110.244, 99.9467, 74.3863, 38.3165, 22.1387, 13.8638, 9.20954762864289, 6.40746485225397,
    4.62692014996195, 2.30743153037744, 1.61505075253763, 1.30130979037554, 0.801145676760343,
    0.525990443101652, 0.262079759969996, 0.17213944380102, 0.150643910752649, 0.096166283094939,
    0.067006659158694, 0.040350724533314, 0.030060380204255, 0.025736225862943, 0.023919178948042,
    0.023273564441493, 0.023169137685326, 0.023905175732908, 0.024501233602389, 0.024926643691358,
    0.025877217107796, 0.026274062287156, 0.026655131026205, 0.027882238669658, 0.028683812435718,
    0.028972447929017, 0.029148771809045, 0.029415131704691, 0.029490249846129, 0.029698178702551,
    0.029685041261757, 0.029603764611649, 0.029549718949812, 0.029405101930449, 0.029197657857256,
    0.028890614121642, 0.028390536220701, 0.027936798166738, 0.026719405747041, 0.02562121786982,
    0.025359438553807, 0.02463108715039, 0.024240842040503, 0.023753744772526, 0.022671649894621,
    0.022309768351513, 0.021132688079239, 0.020639133403572, 0.020121558259145, 0.019347638398287,
    0.018700741656237, 0.018185053120065, 0.017326066208925, 0.016966042150857, 0.016690444038446,
    0.016199645451199, 0.015809252632524
]
cmi_2025_energies_h_star_10_0 = [
    2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 22, 24, 26, 28, 30, 32, 34, 36, 38, 40,
    42, 44, 46, 48, 50, 52, 54, 56, 58, 60, 65, 70, 75, 80, 85, 90, 95, 100, 110, 120, 130, 140, 150, 160, 170,
    180, 190, 200, 225, 240, 250, 275, 300, 325, 350, 375, 400, 425, 450, 500, 511, 550, 600, 662, 700, 800,
    900, 1000, 1117, 1173, 1250, 1332, 1500, 1700, 1750, 2000, 2400, 2500, 3000, 3500, 4000, 4440, 5000, 6000,
    6129, 7000, 8000, 9000, 10000, 12500, 15000, 17500, 20000, 25000, 30000, 35000, 40000, 45000, 50000
]
cmi_2025_values_h_star_10_0 = [
    0.0, 0.0, 0.0, 0.0, 0.0, 5.006211183e-07, 7.53411188163e-05, 0.0013399279666727, 0.008339974116065,
    0.027959365493142, 0.0649015031392173, 0.117892384153277, 0.182854496500547, 0.254785065497812,
    0.329684376352027, 0.404237490435794, 0.474883646496535, 0.543075936619701, 0.608764764690923,
    0.729121699136139, 0.834359989357258, 0.932217912526701, 1.019787991814, 1.10127304871649, 1.17923580150499,
    1.25093929821867, 1.31693499152686, 1.37836679008758, 1.43614832274801, 1.48895147535783, 1.53695987736966,
    1.579367978262, 1.61683555911064, 1.65111364109003, 1.67895036865187, 1.70147532324998, 1.71959205780654,
    1.73569389238546, 1.7513548512875, 1.7660601751939, 1.76411105061724, 1.75880008458157, 1.74401588142437,
    1.7259083944068, 1.7051120882291, 1.68285949300524, 1.660806895852, 1.62069771752482, 1.58655624975128,
    1.55253890207103, 1.5214025670048, 1.49292559713724, 1.47191391445754, 1.45095250563087, 1.43234388256437,
    1.41504458319957, 1.40045048056932, 1.36708755516683, 1.35062396617274, 1.34112411144134, 1.31829850613254,
    1.29920469418009, 1.28280551309179, 1.26779645957562, 1.25550014309865, 1.24492592905825, 1.23453216595746,
    1.225707899164, 1.2113467185313, 1.20789939579853, 1.19806539226821, 1.18818337262196, 1.17653859236809,
    1.17219827416785, 1.16041231542684, 1.14984262965137, 1.14263898967494, 1.13414788332302, 1.13119321285273,
    1.12823105732118, 1.12464830688625, 1.11896627040591, 1.11357509036097, 1.11227237543666, 1.1077193167192,
    1.10088553317439, 1.09989918128124, 1.09354396763082, 1.08802563926124, 1.08425634507853, 1.07995815186229,
    1.07712879613375, 1.0702298390959, 1.06941454173716, 1.06406552317947, 1.05908367593344, 1.05411440308156,
    1.05022231792777, 1.04082044834368, 1.03355131530843, 1.02708988121954, 1.02235595808979, 1.01449765630958,
    1.00889446838262, 1.00286377763245, 1.00129856074053, 0.997193631804752, 0.996021011235643
]

# Quality definitions
name_1 = "N60"
voltage_1 = 60
total_filtration_1 = {"Al": 4, "Cu": 0.6}

# Spekpy calculations
s = sp.Spek(kvp=voltage_1, th=20)
s.multi_filter([["Al", 4.0], ["Cu", 0.6], ["Air", 1000]]) # TODO: See Note 1

spectrum_1 = s.get_spectrum(diff=False) # See Note 2
e_mean_1 = s.get_emean()
kerma_1 = s.get_kerma()
hvl1_al_1 = s.get_hvl1()
hvl2_al_1 = s.get_hvl2()
hvl1_cu_1 = s.get_hvl1(matl="Cu")
hvl2_cu_1 = s.get_hvl2(matl="Cu")

# Mean conversion coefficient calculation

# Getting coefficients data
spec_energies = np.array(spectrum_1[0])
spec_values = np.array(spectrum_1[1])
mu_energies = np.array(pene_2018_energies)
mu_values = np.array(pene_2018_values)
h_k_energies = np.array(cmi_2025_energies_h_star_10_0)
h_k_values = np.array(cmi_2025_values_h_star_10_0)

# If there are zeros in mu_tr_over_rho_air or h_k values, we need to avoid log(0)
# We can remove the zeros from the coefficients
if np.any(mu_values == 0):
    mask_mu = mu_values != 0
    filtered_mu_energies = mu_energies[mask_mu]
    filtered_mu_values = mu_values[mask_mu]
    print("Warning: Zeros found in mu_tr_over_rho_air values. They have been removed for interpolation.")
else:
    filtered_mu_energies = mu_energies
    filtered_mu_values = mu_values

if np.any(h_k_values == 0):
    mask_hk = h_k_values != 0
    filtered_hk_energies = h_k_energies[mask_hk]
    filtered_hk_values = h_k_values[mask_hk]
    print("Warning: Zeros found in h_k values. They have been removed for interpolation.")
else:
    filtered_hk_energies = h_k_energies
    filtered_hk_values = h_k_values

# Interpolating mu_tr_over_rho_air coefficients to spectrum energies
interpolator = Akima1DInterpolator(x=np.log(filtered_mu_energies), y=np.log(filtered_mu_values))
interpolated_mu = np.exp(interpolator(np.log(spec_energies)))

# Interpolating h_k coefficients to spectrum energies
interpolator = Akima1DInterpolator(x=np.log(filtered_hk_energies), y=np.log(filtered_hk_values))
interpolated_hk = np.exp(interpolator(np.log(spec_energies)))

# Check for NaN values in interpolated results
if np.any(np.isnan(interpolated_mu)):
    print("Warning: NaN values found in interpolated mu_tr_over_rho_air.")
if np.any(np.isnan(interpolated_hk)):
    print("Warning: NaN values found in interpolated h_k.")

# Calculating mean conversion coefficient
h_k_mean_numerator = np.nansum(spec_values * spec_energies * interpolated_mu * interpolated_hk)
h_k_mean_denominator = np.nansum(spec_values * spec_energies * interpolated_mu)
h_k_mean_1 = h_k_mean_numerator / h_k_mean_denominator

### Using Quality class from metpyx.sim

In [ ]:
# Using Quality class from metpyx.sim

# Quality definitions
quality = Quality("N60", th=20)
name_2 = quality.quality
voltage_2 = quality.voltage
total_filtration_2 = quality.total_filtration

# Spekpy calculations
spectrum_2 = quality.get_spectrum(diff=False)
e_mean_2 = quality.get_emean()
kerma_2 = quality.get_kerma()
hvl1_al_2 = quality.get_hvl1()
hvl2_al_2 = quality.get_hvl2()
hvl1_cu_2 = quality.get_hvl1(matl="Cu")
hvl2_cu_2 = quality.get_hvl2(matl="Cu")

# Calculating mean conversion coefficient
h_k_mean_2 = quality.get_hk_mean('h_star_10', 0)

### Comparison

In [ ]:
# Assertions for verification

# Quality definitions
assert name_1 == name_2
assert voltage_1 == voltage_2
assert total_filtration_1 == total_filtration_2

# Spekpy calculations
assert list(spectrum_1[0]) == list(spectrum_2[0])
assert list(spectrum_1[1]) == list(spectrum_2[1]), f"Spectrum 1\n{list(spectrum_1[1])}\nSpectrum 2\n{list(spectrum_2[1])}."
assert e_mean_1 == e_mean_2
assert kerma_1 == kerma_2
assert hvl1_al_1 == hvl1_al_2
assert hvl2_al_1 == hvl2_al_2
assert hvl1_cu_1 == hvl1_cu_2
assert hvl2_cu_1 == hvl2_cu_2

# Mean conversion coefficient calculation
assert h_k_mean_1 == h_k_mean_2

print("All verifications passed.")

## Note 1: On SpekPy air thickness and distance
How to calculate exactly at 1 m and 2.5 m?
To be precise, air thickness should be measurement distance - total filtration thickness.
There is a parameter in spekpy to set measurement distance.
In order to compare with previous results I am not going  to change the approach, but I should look into this later.

In [ ]:
# On SpekPy air thickness and distance

s1 = sp.Spek(kvp=60)
s1.multi_filter([["Al", 4.0], ["Cu", 0.6], ["Air", 1000]])
kerma1 = s1.get_kerma()

s2 = sp.Spek(kvp=60, z=100)
s2.multi_filter([["Al", 4.0], ["Cu", 0.6], ["Air", 1000]])
kerma2 = s2.get_kerma()

s3 = sp.Spek(kvp=60, z=250)
s3.multi_filter([["Al", 4.0], ["Cu", 0.6], ["Air", 2500]])
kerma3 = s3.get_kerma()

s4 = sp.Spek(kvp=60)
s4.multi_filter([["Al", 4.0], ["Cu", 0.6], ["Air", 2500]])
kerma4 = s4.get_kerma()

s5 = sp.Spek(kvp=60)
distance = s5.state.spectrum_parameters.z
total_filtration_thickness = sum(item[1] for item in [["Al", 4.0], ["Cu", 0.6]])
air_thickness1 = distance*10 - total_filtration_thickness
s5.multi_filter([["Al", 4.0], ["Cu", 0.6], ["Air", air_thickness1]])
kerma5 = s5.get_kerma()

s6 = sp.Spek(kvp=60, z=250)
distance = s6.state.spectrum_parameters.z
total_filtration_thickness = sum(item[1] for item in [["Al", 4.0], ["Cu", 0.6]])
air_thickness2 = distance*10 - total_filtration_thickness
s6.multi_filter([["Al", 4.0], ["Cu", 0.6], ["Air", air_thickness2]])
kerma6 = s6.get_kerma()

print('Default distance 100?', kerma1, kerma2)
print('Distance 100 vs 250 using z parameter:', kerma1, kerma3)
print('Distance 100 vs 250 using air filter:', kerma1, kerma4)
print(f'Air thickness equal to {1000} vs {air_thickness1}:', kerma1, kerma5)
print(f'Air thickness equal to {2500} vs {air_thickness2}:', kerma3, kerma6)

In [ ]:
# Calculating kerma at 1 m and 2.5 m

q1 = Quality("N60")
kerma7 = q1.get_kerma()

q2 = Quality("N60", z=250)
kerma8 = q2.get_kerma()

print('Kerma at 1 m, spekpy vs metpyx:', kerma1, kerma7)
print('Kerma at 2.5 m, spekpy vs metpyx:', kerma3, kerma8)

In [ ]:
# Calculating mean conversion coefficient at 1 m and 2.5 m
q100 = Quality("N60", th=20, z=100)
q250 = Quality("N60", th=20, z=250)

mean_hk_100 = q100.get_hk_mean('h_star_10', 0)
mean_hk_250 = q250.get_hk_mean('h_star_10', 0)

print('Mean h_k at 1 m vs 2.5 m:', mean_hk_100, mean_hk_250, abs(mean_hk_250-mean_hk_100)/mean_hk_100, '% difference')

## Note 2: On SpekPy spectrum parameters
To correctly calculate the mean conversion coefficient, the parameters for spekpy get_spectrum() method must be configured with edges=False, flu=True, diff=False, sig=False. This means energy bins are defined at the mid-bin values, fluence is returned, returned fluence is non-differential in energy (1/cm²), the calculated spectrum is returned without smoothing. Only the diff parameter is different from the default value (see method API in https://bitbucket.org/spekpy/spekpy_release/wiki/Function%20glossary).

## Verification against previous calculations

### MetPyX Quality class vs independent calculation, USpekPy and A1.1.7 report for all qualities and quantities

In [ ]:
# Calculations with USpekPy
# Define x-ray beam parameters for radiation quality N-60 (filter thickness, peak kilovoltage and anode angle)
my_filters = [
  ('Al', 4),  # mm
  ('Cu', 0.6),  # mm
  ('Sn', 0),  # mm
  ('Pb', 0),  # mm
  ('Be', 0),  # mm
  ('Air', 1000)  # mm
]
my_kvp = 60  # kV
my_th = 20  # deg

# Define path to CSV file containing coefficients
my_mu_csv = '/home/u6406/PycharmProjects/metpyx/src/metpyx/data/tables/mu_tr_over_rho_air/pene_2018.csv'
my_hk_csv = '/home/u6406/PycharmProjects/metpyx/src/metpyx/data/tables/h_k/cmi_2025/h_star_10.csv'

df = pd.read_csv(my_hk_csv)
my_hk_e, my_hk_v =(np.array(df['E keV']), np.array(df['hK 0 Sv/Gy']))

mask_hk = my_hk_v != 0
filtered_hk_energies = my_hk_e[mask_hk]
filtered_hk_values = my_hk_v[mask_hk]

# Initialize an SpeckWrapper object and add filters
spectrum = usp.SpekWrapper(kvp=my_kvp, th=my_th)
spectrum.multi_filter(my_filters)

# Calculate half-value layers for aluminum and copper
hvl1_al_3 = spectrum.get_hvl1()
hvl2_al_3 = spectrum.get_hvl2()
hvl1_cu_3 = spectrum.get_hvl1(matl='Cu')
hvl2_cu_3 = spectrum.get_hvl2(matl='Cu')
e_mean_3 = spectrum.get_mean_energy()
kerma_3 = spectrum.get_air_kerma(mass_transfer_coefficients=my_mu_csv)
h_k_mean_3 = spectrum.get_mean_conversion_coefficient(mass_transfer_coefficients=my_mu_csv, conversion_coefficients=(filtered_hk_energies, filtered_hk_values))

In [ ]:
# Gather results
independent = {'hvl1_al': hvl1_al_1, 'hvl2_al': hvl2_al_1, 'hvl1_cu': hvl1_cu_1, 'hvl2_cu': hvl2_cu_1,
               'e_mean': e_mean_1, 'kerma': kerma_1, 'hk_mean': h_k_mean_1}
quality_class = {'hvl1_al': hvl1_al_2, 'hvl2_al': hvl2_al_2, 'hvl1_cu': hvl1_cu_2, 'hvl2_cu': hvl2_cu_2,
               'e_mean': e_mean_2, 'kerma': kerma_2, 'hk_mean': h_k_mean_2}
uspekpy = {'hvl1_al': hvl1_al_3, 'hvl2_al': hvl2_al_3, 'hvl1_cu': hvl1_cu_3, 'hvl2_cu': hvl2_cu_3,
               'e_mean': e_mean_3, 'kerma': kerma_3, 'hk_mean': h_k_mean_3}
a117 = {'hvl1_al': 5.8582747051770605, 'hvl2_al': 6.196968255804022, 'hvl1_cu': 0.23236896040944352,
        'hvl2_cu': 0.25978939890001607, 'e_mean': 47.63300484905786, 'kerma': 1.5821240129760867,
        'hk_mean': 1.5652667635344528}

In [ ]:
# Assertions for verification
try:
    assert quality_class == independent, f"Quality class results:\n{quality_class}\nIndependent calculation results:\n{independent}"
    assert quality_class == uspekpy, f"Quality class results:\n{quality_class}\nUSpekPy results:\n{uspekpy}"
    assert quality_class == a117, f"Quality class results:\n{quality_class}\nCIEMAT A1.1.7 report results:\n{a117}"
except AssertionError as exc:
    warnings.simplefilter('always', UserWarning)           # ensure warning is shown
    warnings.warn(str(exc), UserWarning, stacklevel=2)

In [ ]:
# Build DataFrame for comparison
df1 = pd.DataFrame({'Independent': independent, 'Quality class': quality_class, 'A117': a117, 'USpekPy': uspekpy})
rel_diff1 = df1.sub(df1['Quality class'], axis=0).div(df1['Quality class'], axis=0).mul(100)

In [ ]:
# Print values comparison
print(df1)

In [ ]:
# Print relative differences
print(rel_diff1)

In [ ]:
# Plot values comparison
df1.plot(marker='o')

In [ ]:
# Plot relative differences
rel_diff1.plot(marker='o')

### MetPyX Quality class vs reported values in CIEMAT A1.1.7 for selected qualities and quantities
Comparison between MetPyX (Quality class) and reported values in CIEMAT A1.1.7 for selected qualities and quantities (Jaroslav Excel in email *"GuideRadPROS A1.1.7: preliminary results for A1.1.7 - kV variations"*).

In [ ]:
# Calculations with Quality class for selected qualities and quantities
q1 = Quality("N30", th=20)
q2 = Quality("N40", th=20)
q3 = Quality("N60", th=20)
q4 = Quality("H60", th=20)
q5 = Quality("N250", th=20)

metpyx = {
    ('N30', 'h_star_10', 0): q1.get_hk_mean('h_star_10', 0),
    ('N30', 'h_p_07_slab', 0): q1.get_hk_mean('h_p_07_slab', 0),
    ('N30', 'h_p_07_rod', 0): q1.get_hk_mean('h_p_07_rod', 0),
    ('N40', 'h_star_10', 0): q2.get_hk_mean('h_star_10', 0),
    ('N40', 'h_p_07_slab', 0): q2.get_hk_mean('h_p_07_slab', 0),
    ('N40', 'h_p_07_rod', 0): q2.get_hk_mean('h_p_07_rod', 0),
    ('N60', 'h_star_10', 0): q3.get_hk_mean('h_star_10', 0),
    ('N60', 'h_p_07_slab', 0): q3.get_hk_mean('h_p_07_slab', 0),
    ('N60', 'h_p_07_rod', 0): q3.get_hk_mean('h_p_07_rod', 0),
    ('H60', 'h_star_10', 0): q4.get_hk_mean('h_star_10', 0),
    ('H60', 'h_p_07_slab', 0): q4.get_hk_mean('h_p_07_slab', 0),
    ('H60', 'h_p_07_rod', 0): q4.get_hk_mean('h_p_07_rod', 0),
    ('N250', 'h_star_10', 0): q5.get_hk_mean('h_star_10', 0),
    ('N250', 'h_p_07_slab', 0): q5.get_hk_mean('h_p_07_slab', 0),
    ('N250', 'h_p_07_rod', 0): q5.get_hk_mean('h_p_07_rod', 0),
}

In [ ]:
# Reported values in CIEMAT A1.1.7 for selected qualities and quantities
reported_ciemat_a117 = {
    ('N30', 'h_star_10', 0): 0.806885558316363,
    ('N30', 'h_p_07_slab', 0): 1.11448855756614,
    ('N30', 'h_p_07_rod', 0): 1.04324004559772,
    ('N40', 'h_star_10', 0): 1.17593033663222,
    ('N40', 'h_p_07_slab', 0): 1.28317484599388,
    ('N40', 'h_p_07_rod', 0): 1.07520090386855,
    ('N60', 'h_star_10', 0): 1.5651644658369,
    ('N60', 'h_p_07_slab', 0): 1.55514905850875,
    ('N60', 'h_p_07_rod', 0): 1.11462960540175,
    ('H60', 'h_star_10', 0): 1.1916262091482,
    ('H60', 'h_p_07_slab', 0): 1.31396159889724,
    ('H60', 'h_p_07_rod', 0): 1.07785061763632,
    ('N250', 'h_star_10', 0): 1.38786367074322,
    ('N250', 'h_p_07_slab', 0): 1.42476508709555,
    ('N250', 'h_p_07_rod', 0): 1.14923205051759,
}

In [ ]:
# Assertions for verification
try:
    assert metpyx == reported_ciemat_a117, f"MetPyX results:\n{metpyx}\nReported CIEMAT A1.1.7 values:\n{reported_ciemat_a117}"
except AssertionError as exc:
    warnings.simplefilter('always', UserWarning)           # ensure warning is shown
    warnings.warn(str(exc), UserWarning, stacklevel=2)

In [ ]:
# Creating a DataFrame for comparison and calculating relative differences
df2 = pd.DataFrame({'MetPyX': metpyx, 'Reported CIEMAT A1.1.7': reported_ciemat_a117})
df2['Relative difference (%)'] = (df2['MetPyX']- df2['Reported CIEMAT A1.1.7']) / df2['Reported CIEMAT A1.1.7'] * 100

In [ ]:
# Maximum relative difference
print('Maximum relative difference:', df2['Relative difference (%)'].abs().max(), '%')

In [ ]:
# Print values and relative differences
print(df2)

In [ ]:
# Plot values comparison
df2[['MetPyX', 'Reported CIEMAT A1.1.7']].plot(marker='o', rot=90)

In [ ]:
# Plot relative differences
df2[['Relative difference (%)']].plot(marker='o', rot=90)

## Verification of all qualities

### Integral quantities independent from operational quantities

In [ ]:
# Using Quality class

qs = Qualities()
qualities = qs.get_all_qualities()
metpyx_results = {}
for quality in qualities:
    q = Quality(quality, th=20)
    metpyx_results[quality] = {
        'e_mean': q.get_emean(),
        'kerma': q.get_kerma(),
        'hvl1_al': q.get_hvl1(),
        'hvl2_al': q.get_hvl2(),
        'hvl1_cu': q.get_hvl1(matl="Cu"),
        'hvl2_cu': q.get_hvl2(matl="Cu")
    }

In [ ]:
# Using SpekPy
total_filtration = {
    'L10': {'Be': 1, 'Al': 0.3},
    'L20': {'Be': 1, 'Al': 2},
    'L30': {'Be': 1, 'Cu': 0.18, 'Al': 4},
    'L35': {'Al': 4, 'Cu': 0.25},
    'L55': {'Al': 4, 'Cu': 1.2},
    'L70': {'Al': 4, 'Cu': 2.5},
    'L100': {'Al': 4, 'Cu': 0.5, 'Sn': 2},
    'L125': {'Al': 4, 'Cu': 1, 'Sn': 4},
    'L170': {'Al': 4, 'Cu': 1, 'Sn': 3, 'Pb': 1.5},
    'L210': {'Al': 4, 'Cu': 0.5, 'Sn': 2, 'Pb': 3.5},
    'L240': {'Al': 4, 'Cu': 0.5, 'Sn': 2, 'Pb': 5.5},
    'N10': {'Be': 1, 'Al': 0.1},
    'N15': {'Be': 1, 'Al': 0.5},
    'N20': {'Be': 1, 'Al': 1},
    'N25': {'Be': 1, 'Al': 2},
    'N30': {'Be': 1, 'Al': 4},
    'N40': {'Al': 4, 'Cu': 0.21},
    'N60': {'Al': 4, 'Cu': 0.6},
    'N80': {'Al': 4, 'Cu': 2},
    'N100': {'Al': 4, 'Cu': 5},
    'N120': {'Al': 4, 'Sn': 1, 'Cu': 5},
    'N150': {'Al': 4, 'Sn': 2.5},
    'N200': {'Al': 4, 'Sn': 3, 'Pb': 1, 'Cu': 2},
    'N250': {'Al': 4, 'Sn': 2, 'Pb': 3, },
    'N300': {'Al': 4, 'Sn': 3, 'Pb': 5, },
    'N350': {'Al': 4, 'Sn': 4.5, 'Pb': 7, },
    'N400': {'Al': 4, 'Sn': 6, 'Pb': 10, },
    'W30': {'Be': 1, 'Al': 2},
    'W40': {'Be': 1, 'Al': 4},
    'W60': {'Al': 4, 'Cu': 0.3},
    'W80': {'Al': 4, 'Cu': 0.5},
    'W110': {'Al': 4, 'Cu': 2},
    'W150': {'Al': 4, 'Sn': 1},
    'W200': {'Al': 4, 'Sn': 2},
    'W250': {'Al': 4, 'Sn': 4},
    'W300': {'Al': 4, 'Sn': 6.5},
    'H10': {'Be': 1},
    'H20': {'Be': 1, 'Al': 0.15},
    'H30': {'Be': 1, 'Al': 0.5},
    'H40': {'Be': 1, 'Al': 1.0},
    'H60': {'Be': 1, 'Al': 3.9},
    'H80': {'Al': 7.2},
    'H100': {'Al': 4, 'Cu': 0.15},
    'H150': {'Al': 4, 'Cu': 0.5},
    'H200': {'Al': 4, 'Cu': 1},
    'H250': {'Al': 4, 'Cu': 1.6},
    'H280': {'Al': 4, 'Cu': 3},
    'H300': {'Al': 4, 'Cu': 2.2},
    'H350': {'Al': 4, 'Cu': 3.4},
    'H400': {'Al': 4, 'Cu': 4.7},
}
spekpy_results = {}
for quality, filtration in total_filtration.items():
    s = sp.Spek(kvp=int(quality[1:]), th=20)
    filter_list = [[material, thickness] for material, thickness in filtration.items()]
    filter_list.append(['Air', 1000])
    s.multi_filter(filter_list)
    spekpy_results[quality] = {
        'e_mean':  s.get_emean(),
        'kerma':   s.get_kerma(),
        'hvl1_al': s.get_hvl1(),
        'hvl2_al': s.get_hvl2(),
        'hvl1_cu': s.get_hvl1(matl="Cu"),
        'hvl2_cu': s.get_hvl2(matl="Cu")
    }

In [ ]:
# Assertion
assert metpyx_results == spekpy_results, f"MetPyX results:\n{metpyx_results}\n\nSpekPy results:\n{spekpy_results}"

In [ ]:
# Build dataframes for plotting
df_metpyx = pd.DataFrame(metpyx_results).T
df_spekpy = pd.DataFrame(spekpy_results).T
rel_diff2 = (df_metpyx - df_spekpy).div(df_spekpy)

In [ ]:
# Maximum relative difference
print('Maximum relative difference:', rel_diff2.abs().max(), '%')

In [ ]:
# Plot relative differences
rel_diff2.plot(marker='o', rot=90)

In [ ]:
# Plot mean energy comparison
fig, ax = plt.subplots(figsize=(15, 5))
ax.plot(df_metpyx.index, df_spekpy['e_mean'], label='spekpy', marker='s', linestyle='-')
ax.plot(df_metpyx.index, df_metpyx['e_mean'], label='metpyx', marker='o', linestyle='--')
ax.set_xlabel('Index')
ax.set_ylabel(r'$\overline{E}$ (keV)')
ax.tick_params(axis='x', rotation=90)
ax.grid(True)
ax.legend()
plt.show()

In [ ]:
# Plot air kerma comparison
fig, ax = plt.subplots(figsize=(15, 5))
ax.plot(df_metpyx.index, df_spekpy['kerma'], label='spekpy', marker='s', linestyle='-')
ax.plot(df_metpyx.index, df_metpyx['kerma'], label='metpyx', marker='o', linestyle='--')
ax.set_xlabel('Index')
ax.set_ylabel(r'$K_{air}$ ($\mu$Gy)') # TODO: Check units
ax.tick_params(axis='x', rotation=90)
ax.grid(True)
ax.legend()
plt.show()

In [ ]:
# Plot HVL1 Al comparison
fig, ax = plt.subplots(figsize=(15, 5))
ax.plot(df_metpyx.index, df_spekpy['hvl1_al'], label='spekpy', marker='s', linestyle='-')
ax.plot(df_metpyx.index, df_metpyx['hvl1_al'], label='metpyx', marker='o', linestyle='--')
ax.set_xlabel('Index')
ax.set_ylabel(r'HVL1 Al (mm)')
ax.tick_params(axis='x', rotation=90)
ax.grid(True)
ax.legend()
plt.show()

In [ ]:
# Plot HVL2 Al comparison
fig, ax = plt.subplots(figsize=(15, 5))
ax.plot(df_metpyx.index, df_spekpy['hvl2_al'], label='spekpy', marker='s', linestyle='-')
ax.plot(df_metpyx.index, df_metpyx['hvl2_al'], label='metpyx', marker='o', linestyle='--')
ax.set_xlabel('Index')
ax.set_ylabel(r'HVL2 Al (mm)')
ax.tick_params(axis='x', rotation=90)
ax.grid(True)
ax.legend()
plt.show()

In [ ]:
# Plot HVL1 Cu comparison
fig, ax = plt.subplots(figsize=(15, 5))
ax.plot(df_metpyx.index, df_spekpy['hvl1_cu'], label='spekpy', marker='s', linestyle='-')
ax.plot(df_metpyx.index, df_metpyx['hvl1_cu'], label='metpyx', marker='o', linestyle='--')
ax.set_xlabel('Index')
ax.set_ylabel(r'HVL1 Cu (mm)')
ax.tick_params(axis='x', rotation=90)
ax.grid(True)
ax.legend()
plt.show()

In [ ]:
# Plot HVL2 Cu comparison
fig, ax = plt.subplots(figsize=(15, 5))
ax.plot(df_metpyx.index, df_spekpy['hvl2_cu'], label='spekpy', marker='s', linestyle='-')
ax.plot(df_metpyx.index, df_metpyx['hvl2_cu'], label='metpyx', marker='o', linestyle='--')
ax.set_xlabel('Index')
ax.set_ylabel(r'HVL2 Cu (mm)')
ax.tick_params(axis='x', rotation=90)
ax.grid(True)
ax.legend()
plt.show()

### Mean conversion coefficient for all qualities and quantities

In [ ]:
# Using Quality class

qls = Qualities()
qts = OperationalQuantities()
qualities = qls.get_all_qualities()
quantities = qts.get_all_quantities()

metpyx_results = {}
for quality in qualities:
    q = Quality(quality, th=20)
    metpyx_results[quality] = {}
    for quantity in quantities:
        for angle in qts.get_irradiation_angles(quantity):
            # print(quality, quantity, angle)
            metpyx_results[quality][(f'{quantity}', f'{angle}')] = q.get_hk_mean(quantity, angle)

In [ ]:
# Build DataFrame
df_metpyx = pd.DataFrame(metpyx_results).T

In [ ]:
# From A1.1.7 report for all qualities and quantities
a117_file = '/home/u6406/PycharmProjects/metpyx/notebooks/a117_report_all_qq_hk.csv'
df_a117 = pd.read_csv(a117_file, index_col=0)

In [ ]:
# Modifying MetPyX DataFrame for comparison
df_metpyx = pd.DataFrame(metpyx_results).T
df_metpyx = df_metpyx.T.reset_index()
quantities_map = {
    'h_p_07_pill':'Hp(0.07, pillar)',
    'h_p_07_rod':'Hp(0.07, rod)',
    'h_p_07_slab':'Hp(0.07, slab)',
    'h_p_10_slab':'Hp(10, slab)',
    'h_p_3_cyl':'Hp(3, cyl)',
    'h_prime_07':'H_prime(0.07)',
    'h_prime_3':'H_prime(3)',
    'h_star_10':'H*(10)'
}
df_metpyx['level_0']=df_metpyx['level_0'].replace(quantities_map)
df_metpyx['level_1'] = df_metpyx['level_1'].add('º')
df_metpyx['Operational quantity'] = df_metpyx['level_0'].str.cat(df_metpyx['level_1'], sep=' ')
df_metpyx.drop(['level_0', 'level_1'], axis=1, inplace=True)
df_metpyx.set_index('Operational quantity', inplace=True)

assert len(df_metpyx.columns) == len(df_a117.columns)
assert len(df_metpyx.index) == len(df_a117.index)
assert set(df_metpyx.columns) == set(df_a117.columns)
assert set(df_metpyx.index) == set(df_a117.index)

In [ ]:
# Calculate relative differences
rel_diff3 = (df_metpyx - df_a117).div(df_a117)

In [ ]:
# Maximum relative difference
print('Maximum relative difference:', rel_diff3.abs().max(), '%')